## Working with TiTiler-EoPF - Mosaic

This notebook demonstrates how to use the TiTiler-EoPF service to visualize a **collection** (STAC) of EOPF Zarr datastores.


In [ ]:
# Start titiler-eopf services locally with Docker

!docker compose up api -d

In [5]:
import json
import httpx2 as httpx
from folium import Map, TileLayer

%matplotlib inline

In [6]:
titiler_endpoint = "http://127.0.0.1:8000"

In [8]:
r = httpx.get(f"{titiler_endpoint}/_mgmt/health")
print(r.json())

{'status': 'UP', 'versions': {'titiler': '0.10.1', 'rasterio': '1.5.1', 'gdal': '3.12.4', 'proj': '9.8.1', 'geos': '0.0.0', 'xarray': '2026.7.0', 'zarr': '3.3.0'}}


### Conformances

In [9]:
r = httpx.get(f"{titiler_endpoint}/conformance").json()
print(json.dumps(r, indent=4))

{
    "conformsTo": [
        "http://www.opengis.net/spec/ogcapi-common-1/1.0/conf/core",
        "http://www.opengis.net/spec/ogcapi-common-1/1.0/conf/html",
        "http://www.opengis.net/spec/ogcapi-common-1/1.0/conf/json",
        "http://www.opengis.net/spec/ogcapi-common-1/1.0/conf/landing-page",
        "http://www.opengis.net/spec/ogcapi-common-1/1.0/conf/oas30",
        "http://www.opengis.net/spec/ogcapi-tiles-1/1.0/conf/core",
        "http://www.opengis.net/spec/ogcapi-tiles-1/1.0/conf/jpeg",
        "http://www.opengis.net/spec/ogcapi-tiles-1/1.0/conf/png",
        "http://www.opengis.net/spec/ogcapi-tiles-1/1.0/conf/tiff",
        "http://www.opengis.net/spec/ogcapi-tiles-1/1.0/conf/tileset",
        "http://www.opengis.net/spec/ogcapi-tiles-1/1.0/conf/tilesets-list",
        "https://www.opengis.net/spec/ogcapi-maps-1/1.0/conf/core",
        "https://www.opengis.net/spec/ogcapi-maps-1/1.0/conf/crs",
        "https://www.opengis.net/spec/ogcapi-maps-1/1.0/conf/jpeg",
  

In [10]:
collection_id = "sentinel-2-l2a"
bbox = [11.393460776835221, 41.42010137184542, 12.466572328262494, 42.426911995973605]
date = "2026-01-17/2026-01-18"

## Collection Info

In [20]:
# Fetch Metadata for all available variables
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/info",
    params={
        "bbox": "11.393460776835221,41.42010137184542,12.466572328262494,42.426911995973605",
        "datetime": "2026-08-01/2026-08-06",
    },
    timeout=20,
).json()

print(json.dumps(r, indent=4))

{
    "bounds": [
        11.393460776835221,
        41.42010137184542,
        12.466572328262494,
        42.426911995973605
    ],
    "crs": "http://www.opengis.net/def/crs/EPSG/0/4326",
    "renders": {}
}


## Display tiles

### Single Asset

In [22]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/WebMercatorQuad/tilejson.json",
    params={
        "bbox": "11.393460776835221,41.42010137184542,12.466572328262494,42.426911995973605",
        "datetime": "2026-08-01/2026-08-06",
        # "ids": "S2B_MSIL2A_20260731T100559_N0512_R022_T33TTG_20260731T142842",
        "assets": "reflectance|bands=red,green,blue",
        "rescale": "0,1",
        "tilesize": 256,
    },
    timeout=10,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=8
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)
m

{'tilejson': '3.0.0', 'version': '1.0.0', 'scheme': 'xyz', 'tiles': ['http://127.0.0.1:8000/collections/sentinel-2-l2a/tiles/WebMercatorQuad/{z}/{x}/{y}?bbox=11.393460776835221%2C41.42010137184542%2C12.466572328262494%2C42.426911995973605&datetime=2026-08-01%2F2026-08-06&assets=reflectance%7Cbands%3Dred%2Cgreen%2Cblue&rescale=0%2C1&tilesize=256'], 'minzoom': 0, 'maxzoom': 24, 'bounds': [11.393460776835221, 41.42010137184542, 12.466572328262494, 42.426911995973605], 'center': [11.930016552548857, 41.92350668390951, 0], 'raster_layers': {}}
